In [ ]:
'''Use SHAP (SHapley Additive exPlanations) values to explain complex model predictions at both global and
local levels, interpreting feature contributions and interaction effects for transparency and accountability.'''

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import shap

In [ ]:
# load dataset and inspect structure
df = pd.read_csv('global_pharmacy_sales_data.csv')
print(df.shape)
df.head()

In [ ]:
# preprocess data and prepare features and target
df_sample = df.sample(n=5000, random_state=42).copy()
for col in ['category', 'age_group', 'region']:
    df_sample[col] = LabelEncoder().fit_transform(df_sample[col])

features = ['unit_price', 'stock_level', 'expiry_days_remaining', 'covid_flag', 'category', 'age_group', 'region', 'month']
X = df_sample[features]
y = df_sample['units_sold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# train xgboost regressor model
model = XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)
print('train r2:', model.score(X_train, y_train))
print('test r2:', model.score(X_test, y_test))

In [ ]:
# compute shap values for test data
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test.iloc[:100])

In [ ]:
# plot global feature importance bar chart
plt.figure(figsize=(8, 4))
shap.plots.bar(shap_values, show=False)
plt.title('global feature importance')
plt.tight_layout()
plt.show()

In [ ]:
# plot global beeswarm summary for feature contributions
plt.figure(figsize=(8, 4))
shap.plots.beeswarm(shap_values, show=False)
plt.title('global feature contributions summary')
plt.tight_layout()
plt.show()

In [ ]:
# plot local waterfall explanation for a single prediction
plt.figure(figsize=(8, 4))
shap.plots.waterfall(shap_values[0], show=False)
plt.title('local waterfall explanation')
plt.tight_layout()
plt.show()

In [ ]:
# plot feature interaction effect
plt.figure(figsize=(8, 4))
shap.plots.scatter(shap_values[:, 'unit_price'], color=shap_values[:, 'stock_level'], show=False)
plt.title('interaction effect between unit price and stock level')
plt.tight_layout()
plt.show()